In [ ]:
# pip install torch torchvision opencv-python gradio pandas

In [ ]:
# import necessary image processing modules
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [ ]:
# define image transforms

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),

    transforms.RandomHorizontalFlip(),  # data augmentation to prevent overfitting
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05,0.05)),
    transforms.RandomCrop((48, 48)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),

    transforms.ToTensor(),
    transforms.RandomErasing(p=0.25),
    transforms.Normalize(mean=0.5, std=0.5), # normalize pixels

])
val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=0.5, std=0.5)
])
test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=0.5, std=0.5)
])

In [ ]:
# mount the FER-2013 dataset from drive to train on
from google.colab import drive
drive.mount('/content/drive')

import zipfile
with zipfile.ZipFile('/content/drive/MyDrive/emotion_data/archive.zip', 'r') as z:
  z.extractall('/content/fer2013')

Mounted at /content/drive


In [ ]:
# remove junk macosx zip directory
import shutil
shutil.rmtree('/content/fer2013/__MACOSX')

In [ ]:
# load datasets
import torch

train_dataset = datasets.ImageFolder(root='fer2013/archive/train', transform=train_transform)
val_dataset = datasets.ImageFolder(root='fer2013/archive/train', transform=val_transform)

train_ind, val_ind = torch.utils.data.random_split(range(len(train_dataset)), [.8,.2])  # split train and validation by indices

train_dataset = torch.utils.data.Subset(train_dataset, train_ind)
val_dataset = torch.utils.data.Subset(val_dataset, val_ind)
test_dataset = datasets.ImageFolder(root='fer2013/archive/test', transform=test_transform)

In [ ]:
# experiment with images

# import matplotlib.pyplot as plt

# img = train_dataset[28011][0]
# print(train_dataset[28011][1])
# plt.imshow(img, cmap='gray')  # since it's grayscale
# plt.axis('off')
# plt.show()

In [ ]:
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [ ]:
# sanity check
print(train_dataset.dataset.classes)
print(f'training samples: {len(train_dataset)}')
print(f'test samples: {len(test_dataset)}')

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
training samples: 22967
test samples: 7178


In [ ]:
from torch.nn.modules.activation import ReLU
from torch.nn.modules.linear import Linear
import torch.nn as nn


# Build CNN
class ConvolutionalNeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(64, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.Conv2d(128, 256, kernel_size=3, padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        # nn.Dropout(0.3),
        nn.Linear(256*6**2, 256),
        nn.ReLU(),
        nn.Linear(256,7)
    )

  def forward(self, x):
    x = self.features(x)
    return self.classifier(x)

In [ ]:
# run one epoch
def run_one_epoch(model, loss, optimizer, train_loader, device):
  total_loss = 0
  avg_total_loss = 0
  for batch, data in enumerate(train_loader):
    img, target = data
    img, target = img.to(device), target.to(device)
    optimizer.zero_grad()
    output = model(img)
    train_loss = loss(output, target)
    train_loss.backward()
    optimizer.step()
    total_loss += train_loss.item()
    avg_total_loss += train_loss.item()
    if (batch+1) % 100 == 0:
      print(f'Batch: {batch+1} | training loss: {total_loss/100}')
      total_loss = 0
  return avg_total_loss/len(train_loader)

In [ ]:
# train 15 epochs

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loss = nn.CrossEntropyLoss()
model = ConvolutionalNeuralNetwork()
model.to(device)      # send model to gpu for faster training
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

epochs = 30
best_vloss = float("inf")
for epoch in range(epochs):
  model.train(True)
  print(f'Epoch: {epoch+1}')
  avg_loss = run_one_epoch(model, loss, optimizer, train_loader, device)

  running_vloss = 0

  model.eval()

  with torch.no_grad():
    for batch, data in enumerate(val_loader):
      img, target = data
      img, target = img.to(device), target.to(device)
      output = model(img)
      validation_loss = loss(output, target)
      running_vloss+=validation_loss.item()

  avg_vloss = running_vloss/len(val_loader)

  if avg_vloss < best_vloss:
    best_vloss = avg_vloss
    torch.save(model.state_dict(), "emotion_model_best.pt")
    print(f"Saved best model with validation loss: {best_vloss:.4f}")

  scheduler.step(avg_vloss)

  current_lr = optimizer.param_groups[0]['lr']

  print(f"Losses: train: {avg_loss} | validation: {avg_vloss} | learning rate: {current_lr}")


Epoch: 1
Batch: 100 | training loss: 2.0926830697059633
Batch: 200 | training loss: 1.8079991459846496
Batch: 300 | training loss: 1.780867304801941
Saved best model with validation loss: 1.8591
Losses: train: 1.8734067651886794 | validation: 1.8591367562611898 | learning rate: 0.001
Epoch: 2
Batch: 100 | training loss: 1.687733038663864
Batch: 200 | training loss: 1.6202063572406769
Batch: 300 | training loss: 1.570016267299652
Saved best model with validation loss: 1.4417
Losses: train: 1.6079361860466534 | validation: 1.4417049805323283 | learning rate: 0.001
Epoch: 3
Batch: 100 | training loss: 1.4758055436611175
Batch: 200 | training loss: 1.474907763004303
Batch: 300 | training loss: 1.4373684847354888
Saved best model with validation loss: 1.3435
Losses: train: 1.4536897760911904 | validation: 1.3434975862503051 | learning rate: 0.001
Epoch: 4
Batch: 100 | training loss: 1.3930214989185332
Batch: 200 | training loss: 1.362275470495224
Batch: 300 | training loss: 1.36501110672950

In [ ]:
state_dict = torch.load("emotion_model_best.pt", map_location=device)
print(state_dict.keys())

odict_keys(['features.0.weight', 'features.0.bias', 'features.1.weight', 'features.1.bias', 'features.1.running_mean', 'features.1.running_var', 'features.1.num_batches_tracked', 'features.4.weight', 'features.4.bias', 'features.5.weight', 'features.5.bias', 'features.5.running_mean', 'features.5.running_var', 'features.5.num_batches_tracked', 'features.7.weight', 'features.7.bias', 'features.8.weight', 'features.8.bias', 'features.8.running_mean', 'features.8.running_var', 'features.8.num_batches_tracked', 'features.11.weight', 'features.11.bias', 'features.12.weight', 'features.12.bias', 'features.12.running_mean', 'features.12.running_var', 'features.12.num_batches_tracked', 'features.14.weight', 'features.14.bias', 'features.15.weight', 'features.15.bias', 'features.15.running_mean', 'features.15.running_var', 'features.15.num_batches_tracked', 'features.17.weight', 'features.17.bias', 'features.18.weight', 'features.18.bias', 'features.18.running_mean', 'features.18.running_var', 

In [ ]:
# testing
model = ConvolutionalNeuralNetwork().to(device)
model.load_state_dict(torch.load("emotion_model_best.pt", map_location=device))
model.eval()

loss_func = nn.CrossEntropyLoss()

correct = 0
total = 0
running_loss = 0

with torch.no_grad():
  for batch, data in enumerate(test_loader):
    img, target = data
    img, target = img.to(device), target.to(device)

    output = model(img)
    loss = loss_func(output, target)

    running_loss += loss.item()

    predicted = torch.argmax(output, dim=1)

    correct += (predicted == target).sum().item()
    total += target.size(0)

avg_loss = running_loss/len(test_loader)
accuracy = correct/total

print('avg loss' + str(avg_loss))
print('accuracy' + str(accuracy))

avg loss0.9812921533542397
accuracy0.6444692114795207
